# Notebook 03: Fabrication & Omission Analysis — LangGraph RAG Pipeline

Runs the LangGraph RAG extraction pipeline on every patient case where the original
AI model produced a fabrication (label=3) or omission (label=2), comparing the
RAG pipeline's output against human reviewer labels.

## Workflow
```
Validation sheet → find fab/omission cases (n=63)
  ↓ MRN × surgeon → case_folder mapping
  ↓ collect all PDFs per patient (471 total)
  ↓ OCR each PDF with caching (pytesseract + fitz, parallel)
  ↓ concatenate PDFs per patient
  ↓ LangGraph RAG pipeline per patient
  ↓ compare verdicts vs human labels
  ↓ figures + auditable JSON output
```

**Input data (private, not in repo):**
- `C:\Users\jamesr4\loc\data_private\raw\merged_llm_summary_validation_datasheet_identified.xlsx`
- `C:\Users\jamesr4\loc\data_private\breast_bot_deidentified\case_id_mapping.csv`
- `C:\Users\jamesr4\loc\data_private\breast_bot_deidentified\{surgeon}\{case_folder}\*.pdf`

## 0. Environment

In [ ]:
import os, sys, json, re, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fitz
import pytesseract
from PIL import Image
from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv()

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(
    r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center"
    r"\Documents\GitHub\llm_summarization_br_ca"
)
DATA_PRIVATE   = Path(r"C:\Users\jamesr4\loc\data_private")
PDF_ROOT       = DATA_PRIVATE / "breast_bot_deidentified"
VALIDATION_XLS = DATA_PRIVATE / "raw" / "merged_llm_summary_validation_datasheet_identified.xlsx"
MAPPING_CSV    = PDF_ROOT / "case_id_mapping.csv"
OCR_CACHE_DIR  = DATA_PRIVATE / "ocr_cache"
RUN_OUT_DIR    = PROJECT_ROOT / "experiments" / "runs" / "fab_omission"
REPORTS_DIR    = PROJECT_ROOT / "reports"

for d in [OCR_CACHE_DIR, RUN_OUT_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

# ── Tesseract ─────────────────────────────────────────────────────────────────
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\jamesr4\AppData\Local\miniforge3\Library\bin\tesseract.exe"
)
os.environ["TESSDATA_PREFIX"] = (
    r"C:\Users\jamesr4\AppData\Local\miniforge3\share\tessdata"
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"PDF_ROOT     : {PDF_ROOT}")
print(f"OCR cache    : {OCR_CACHE_DIR}")
print(f"API key set  : {'ANTHROPIC_API_KEY' in os.environ}")

## 1. Load Validation Sheet — Identify Fabrications & Omissions

In [ ]:
df_val = pd.read_excel(VALIDATION_XLS)

AI_COLS = [c for c in df_val.columns if c.endswith("_status_ai")]
for c in AI_COLS:
    df_val[c] = pd.to_numeric(df_val[c], errors="coerce")

# 2 = omission, 3 = fabrication
fab_mask  = (df_val[AI_COLS] == 3).any(axis=1)
omit_mask = (df_val[AI_COLS] == 2).any(axis=1)
bad_mask  = fab_mask | omit_mask

df_bad = df_val[bad_mask].copy()

print(f"Total cases in sheet : {len(df_val)}")
print(f"Fabrications (3)     : {fab_mask.sum()}  cases")
print(f"Omissions (2)        : {omit_mask.sum()}  cases")
print(f"Either               : {bad_mask.sum()} cases  ← these will be re-run")
print()

# Show which features are most error-prone
feat_errors = pd.DataFrame({
    "feature": [c.replace("_status_ai","") for c in AI_COLS],
    "fabrications": [(df_val[c] == 3).sum() for c in AI_COLS],
    "omissions":    [(df_val[c] == 2).sum() for c in AI_COLS],
}).assign(total=lambda x: x.fabrications + x.omissions).sort_values("total", ascending=False)
print("Error counts by feature:")
print(feat_errors.to_string(index=False))

## 2. Map Cases → PDFs via case_id_mapping.csv

In [ ]:
mapping = pd.read_csv(MAPPING_CSV)

# Extract patient initials embedded in case_folder (position 1 between underscores)
mapping["patient_initials_folder"] = (
    mapping["case_folder"].str.split("_").str[1].str.upper()
)

# Surgeon last-name normalisation
_SURGEON_MAP = {
    "el tamer": "el tamer", "el-tamer": "el tamer",
    "sacchini": "sacchini",
    "giannakou": "giankou", "giankou": "giankou",
    "montagna": "montag",  "montag": "montag",
    "lisa allen": "allen",
}
def _norm(s: str) -> str:
    sl = str(s).strip().lower()
    return _SURGEON_MAP.get(sl, sl)

df_bad = df_bad.copy()
df_bad["surgeon_last"] = df_bad["surgeon"].str.split(",").str[0].str.strip()
df_bad["surgeon_norm"] = df_bad["surgeon_last"].apply(_norm)
mapping["surgeon_norm"] = mapping["surgeon"].apply(_norm)

# Merge: one row per PDF for each bad case
df_merged = df_bad.merge(
    mapping,
    left_on=["surgeon_norm", "patient_initials"],
    right_on=["surgeon_norm", "patient_initials_folder"],
    how="left",
)

unmatched = df_merged["case_id"].isna().sum()
print(f"Patient cases   : {df_bad['mrn'].nunique()}")
print(f"PDF rows        : {len(df_merged)}")
print(f"Unmatched rows  : {unmatched}")
print()

# Build per-patient PDF list  (mrn → [Path, ...])
patient_pdfs: dict = {}
for mrn, grp in df_merged.dropna(subset=["deidentified_path"]).groupby("mrn"):
    pdfs = [Path(p) for p in grp["deidentified_path"].unique() if Path(p).exists()]
    patient_pdfs[int(mrn)] = sorted(pdfs)

pdf_counts = {mrn: len(v) for mrn, v in patient_pdfs.items()}
print(f"Patients with matched PDFs : {len(patient_pdfs)}")
print(f"Total PDFs to process      : {sum(pdf_counts.values())}")
print(f"PDFs/patient range         : {min(pdf_counts.values())}–{max(pdf_counts.values())}")

## 3. OCR Extraction — All PDFs with Caching

In [ ]:
OCR_DPI     = 200   # balance speed vs quality
OCR_WORKERS = 4     # parallel PDF workers
OCR_PSM     = "--psm 6"  # uniform block

def ocr_pdf(pdf_path: Path, dpi: int = OCR_DPI) -> str:
    """OCR one PDF → full text string. Returns cached version if available."""
    cache_file = OCR_CACHE_DIR / (pdf_path.stem + ".txt")
    if cache_file.exists():
        return cache_file.read_text(encoding="utf-8")
    try:
        doc   = fitz.open(str(pdf_path))
        pages = []
        zoom  = dpi / 72.0
        mat   = fitz.Matrix(zoom, zoom)
        for i in range(doc.page_count):
            pix  = doc.load_page(i).get_pixmap(matrix=mat, alpha=False)
            img  = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img, config=OCR_PSM)
            pages.append(f"[PAGE {i+1}]\n{text}")
        doc.close()
        full = "\n\n".join(pages)
        cache_file.write_text(full, encoding="utf-8")
        return full
    except Exception as e:
        return f"[OCR_ERROR: {pdf_path.name}: {e}]"


def ocr_all_pdfs(pdf_list: list, workers: int = OCR_WORKERS) -> dict:
    """OCR a list of PDFs in parallel. Returns {pdf_path: text}."""
    results = {}
    cached  = sum(1 for p in pdf_list if (OCR_CACHE_DIR / (p.stem + ".txt")).exists())
    print(f"  {cached}/{len(pdf_list)} PDFs already cached")

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(ocr_pdf, p): p for p in pdf_list}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="OCR"):
            pdf_path = futures[fut]
            try:
                results[pdf_path] = fut.result()
            except Exception as e:
                results[pdf_path] = f"[FUTURE_ERROR: {e}]"
    return results


# Collect all unique PDFs needed
all_pdfs = sorted({p for pdfs in patient_pdfs.values() for p in pdfs})
print(f"Unique PDFs to OCR: {len(all_pdfs)}")
print("Starting OCR (cached results are instant)...\n")

t0 = time.time()
ocr_results = ocr_all_pdfs(all_pdfs)
elapsed = time.time() - t0
print(f"\nOCR complete in {elapsed/60:.1f} min")
print(f"Chars extracted — median: {int(np.median([len(v) for v in ocr_results.values()]))}, "
      f"max: {max(len(v) for v in ocr_results.values())}")

## 4. Build Per-Patient Concatenated OCR Documents

In [ ]:
# Document ordering priority (imaging before pathology, chronological intent)
_DOC_ORDER = [
    "mammo", "mri", "ultrasound", "US", "imaging",
    "path", "pathology", "receptor", "genetic",
]
def _doc_sort_key(p: Path) -> int:
    name = p.stem.lower()
    for i, kw in enumerate(_DOC_ORDER):
        if kw.lower() in name:
            return i
    return len(_DOC_ORDER)


patient_ocr: dict = {}   # mrn → concatenated OCR text
patient_info: dict = {}  # mrn → {surgeon, patient_initials, case_folder, ...}

for mrn, pdf_paths in patient_pdfs.items():
    sorted_pdfs = sorted(pdf_paths, key=_doc_sort_key)
    sections = []
    for pdf in sorted_pdfs:
        text = ocr_results.get(pdf, "")
        if text and not text.startswith("[OCR_ERROR") and not text.startswith("[FUTURE_ERROR"):
            sections.append(f"=== SOURCE: {pdf.name} ===\n{text}")
    patient_ocr[mrn] = "\n\n".join(sections)

# Store patient metadata for labelling
for _, row in df_bad.drop_duplicates("mrn").iterrows():
    mrn = int(row["mrn"])
    patient_info[mrn] = {
        "id": row["id"],
        "mrn": mrn,
        "patient_initials": row["patient_initials"],
        "surgeon": row["surgeon"],
        "surgeon_last": row["surgeon_last"],
    }

ok_patients = {mrn for mrn, txt in patient_ocr.items() if len(txt) > 200}
print(f"Patients with usable OCR text : {len(ok_patients)} / {len(patient_ocr)}")
sample_mrn = list(ok_patients)[0]
sample_txt = patient_ocr[sample_mrn]
print(f"\nSample MRN {sample_mrn} — {len(sample_txt):,} chars")
print(sample_txt[:500])

## 5. Identify Which Features Need Re-Extraction Per Patient

In [ ]:
# Map validation sheet columns → pipeline feature names
COL_TO_FEATURE = {
    "lesion_size_status_ai":                        "feature_1_lesion_size",
    "laterality_status_ai":                         "feature_2_lesion_location",
    "lesion_location_status_ai":                    "feature_2_lesion_location",
    "calcifications_asymmetry_status_ai":           "feature_3_calcifications_asymmetry",
    "additional_enhancement_mri_status_ai":         "feature_4_additional_enhancement_mri",
    "extent_status_ai":                             "feature_5_extent",
    "accurate_clip_placement_status_ai":            "feature_6_accurate_clip_placement",
    "workup_recommendation_status_ai":              "feature_7_workup_recommendation",
    "Lymph node_status_ai":                         "feature_8_lymph_node",
    "chronology_preserved_status_ai":               "feature_9_chronology_preserved",
    "biopsy_method_status_ai":                      "feature_10_biopsy_method",
    "invasive_component_size_pathology_status_ai":  "feature_11_invasive_component_size_pathology",
    "histologic_diagnosis_status_ai":               "feature_12_histologic_diagnosis",
    "receptor_status_ai":                           "feature_13_receptor_status",
}
# Map human label value → label string
LABEL_DECODE = {1: "CORRECT", 2: "OMISSION", 3: "FABRICATION"}


def get_error_features(row: pd.Series) -> list:
    """Return list of pipeline feature names where AI made an error (2 or 3)."""
    feats = []
    for col, feat in COL_TO_FEATURE.items():
        if col in row.index:
            val = row[col]
            if pd.notna(val) and val in (2, 3):
                feats.append(feat)
    return list(dict.fromkeys(feats))  # deduplicate preserving order


# Build per-patient error map: mrn → {feature: human_label_str}
patient_errors: dict = {}  # mrn → {feature_name: "FABRICATION" | "OMISSION"}
for _, row in df_bad.iterrows():
    mrn = int(row["mrn"])
    errors = {}
    for col, feat in COL_TO_FEATURE.items():
        if col in row.index:
            val = row[col]
            if pd.notna(val) and val in (2, 3):
                errors[feat] = LABEL_DECODE[int(val)]
    patient_errors[mrn] = errors

total_errors = sum(len(v) for v in patient_errors.values())
print(f"Patients with errors   : {len(patient_errors)}")
print(f"Total feature-level errors to re-run: {total_errors}")
print()
# Distribution of errors per patient
n_errs = pd.Series([len(v) for v in patient_errors.values()])
print("Errors per patient:")
print(n_errs.value_counts().sort_index().to_string())

## 6. Run LangGraph RAG Pipeline — All Error Cases

> **Run time estimate:** ~63 patients × avg 3 error features × ~3 API calls/feature ≈ ~570 Claude calls.  
> Estimated cost at claude-3-5-sonnet pricing: ~$2–4 USD.  
> Set `DRY_RUN = True` to skip API calls and use cached results only.

In [ ]:
DRY_RUN = False  # Set True to skip API calls (for testing mapping/OCR only)
MODEL_ID = "claude-3-5-sonnet-20241022"
PROMPT_ID = "rag_verify_v1"

# Load or initialize results cache
RESULTS_CACHE = RUN_OUT_DIR / "pipeline_results.json"
if RESULTS_CACHE.exists():
    with open(RESULTS_CACHE) as f:
        all_results: dict = json.load(f)
    print(f"Loaded {len(all_results)} cached pipeline results from {RESULTS_CACHE}")
else:
    all_results = {}
    print("No cache found — will run from scratch")

print(f"DRY_RUN = {DRY_RUN}")

In [ ]:
from src.workflows.orchestration import run_single_case
from src.utils.io_utils import generate_run_id

run_id = generate_run_id()
print(f"Run ID: {run_id}")

patients_to_run = [
    mrn for mrn in ok_patients
    if str(mrn) not in all_results  # skip already cached
]
print(f"Patients to run: {len(patients_to_run)} (skipping {len(ok_patients) - len(patients_to_run)} cached)")

failed_cases = []

for mrn in tqdm(patients_to_run, desc="Pipeline"):
    ocr_text = patient_ocr[mrn]
    info     = patient_info.get(mrn, {})
    # Only run features that had errors (more efficient than all 13)
    error_feats = list(patient_errors.get(mrn, {}).keys())
    if not error_feats:
        continue

    case_id = f"MRN_{mrn}_{info.get('patient_initials','XX')}"

    if DRY_RUN:
        all_results[str(mrn)] = {
            "case_id": case_id, "mrn": mrn, "dry_run": True,
            "features": {f: {"value": "DRY_RUN", "verdict": None} for f in error_feats},
        }
        continue

    try:
        result = run_single_case(
            case_id=case_id,
            ocr_text=ocr_text,
            prompt_id=PROMPT_ID,
            model_id=MODEL_ID,
            feature_queue=error_feats,  # only re-run error features
            run_id=run_id,
        )
        result["mrn"] = mrn
        result["original_error_features"] = patient_errors.get(mrn, {})
        all_results[str(mrn)] = result

        # Save incrementally after each patient
        with open(RESULTS_CACHE, "w") as f:
            json.dump(all_results, f, indent=2, default=str)

    except Exception as e:
        print(f"  ERROR MRN {mrn}: {e}")
        failed_cases.append({"mrn": mrn, "error": str(e)})

print(f"\nDone. Completed: {len(all_results)}  Failed: {len(failed_cases)}")

## 7. Build Comparison DataFrame — Pipeline vs Human Labels

In [ ]:
VERDICT_COLORS = {
    "CORRECT":     "#2ecc71",
    "FABRICATION": "#e74c3c",
    "OMISSION":    "#f39c12",
    "UNCERTAIN":   "#95a5a6",
    None:          "#bdc3c7",
}

rows = []
for mrn_str, result in all_results.items():
    mrn = int(mrn_str)
    info = patient_info.get(mrn, {})
    orig_errors = result.get("original_error_features", patient_errors.get(mrn, {}))

    for feat, feat_data in result.get("features", {}).items():
        original_label = orig_errors.get(feat)  # "FABRICATION" or "OMISSION"
        if original_label is None:
            continue  # only care about features that had errors

        rows.append({
            "mrn":                     mrn,
            "patient_initials":        info.get("patient_initials"),
            "surgeon":                 info.get("surgeon_last"),
            "feature_name":            feat,
            "original_ai_error":       original_label,
            "pipeline_value":          feat_data.get("value"),
            "pipeline_verdict":        feat_data.get("verdict"),
            "pipeline_confidence":     feat_data.get("confidence"),
            "pipeline_supported":      feat_data.get("supported"),
            "verification_confidence": feat_data.get("verification_confidence"),
            "verification_quote":      feat_data.get("verification_quote"),
            "verification_method":     feat_data.get("verification_method"),
            "retrieval_attempts":      feat_data.get("retrieval_attempts", 0),
            "evidence":                feat_data.get("evidence"),
        })

df_cmp = pd.DataFrame(rows)
print(f"Comparison rows: {len(df_cmp)}")

if not df_cmp.empty:
    print("\nPipeline verdict distribution:")
    print(df_cmp["pipeline_verdict"].value_counts(dropna=False))
    print("\nOriginal AI error distribution:")
    print(df_cmp["original_ai_error"].value_counts())

df_cmp.to_csv(RUN_OUT_DIR / "comparison_results.csv", index=False)
print(f"\nSaved: {RUN_OUT_DIR / 'comparison_results.csv'}")

## 8. Error Recovery Analysis — Did the Pipeline Catch/Fix Errors?

In [ ]:
if df_cmp.empty:
    print("No comparison data yet. Run pipeline first.")
else:
    # Categorise pipeline response to each original error
    def classify_recovery(row):
        orig  = row["original_ai_error"]     # FABRICATION or OMISSION
        pipe  = row["pipeline_verdict"]       # CORRECT, FABRICATION, OMISSION, UNCERTAIN
        if pipe == "CORRECT":
            return "RECOVERED"       # pipeline fixed the original error
        elif pipe == orig:
            return "CONFIRMED_ERROR" # pipeline agreed with original error type
        elif pipe == "UNCERTAIN":
            return "UNCERTAIN"       # pipeline unsure
        elif pipe is None:
            return "NOT_RUN"
        else:
            return "DIFFERENT_ERROR" # different error type

    df_cmp["recovery_status"] = df_cmp.apply(classify_recovery, axis=1)

    print("=== Recovery Status ===")
    print(df_cmp["recovery_status"].value_counts())
    print()

    total = len(df_cmp[df_cmp["recovery_status"] != "NOT_RUN"])
    recovered = (df_cmp["recovery_status"] == "RECOVERED").sum()
    confirmed = (df_cmp["recovery_status"] == "CONFIRMED_ERROR").sum()
    uncertain = (df_cmp["recovery_status"] == "UNCERTAIN").sum()

    print(f"Recovery rate    : {recovered/total:.1%}  ({recovered}/{total})")
    print(f"Confirmed errors : {confirmed/total:.1%}  ({confirmed}/{total})")
    print(f"Uncertain        : {uncertain/total:.1%}  ({uncertain}/{total})")
    print()

    # By original error type
    print("Recovery by original error type:")
    print(
        df_cmp[df_cmp["recovery_status"] != "NOT_RUN"]
        .groupby(["original_ai_error", "recovery_status"])
        .size()
        .unstack(fill_value=0)
        .to_string()
    )

## 9. Figure — Original Error Type vs Pipeline Recovery (Stacked Bar)

In [ ]:
if df_cmp.empty or "recovery_status" not in df_cmp.columns:
    print("Run cell 8 first.")
else:
    rc_colors = {
        "RECOVERED":       "#2ecc71",
        "CONFIRMED_ERROR": "#e74c3c",
        "UNCERTAIN":       "#95a5a6",
        "DIFFERENT_ERROR": "#9b59b6",
        "NOT_RUN":         "#ecf0f1",
    }

    pivot = (
        df_cmp.groupby(["original_ai_error", "recovery_status"])
        .size()
        .unstack(fill_value=0)
    )
    status_order = [s for s in rc_colors if s in pivot.columns]
    pivot = pivot[status_order]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Stacked bar — count
    pivot.plot(
        kind="bar", stacked=True,
        color=[rc_colors[s] for s in status_order],
        ax=axes[0], edgecolor="white", linewidth=1.2,
    )
    axes[0].set_title("Pipeline Response to Original AI Errors\n(by error type)",
                      fontweight="bold")
    axes[0].set_xlabel("Original AI Error")
    axes[0].set_ylabel("Feature Count")
    axes[0].legend(title="Pipeline Result", loc="upper right", fontsize=9)
    axes[0].tick_params(axis="x", rotation=0)

    # Overall donut
    counts = df_cmp["recovery_status"].value_counts()
    wedge_colors = [rc_colors.get(k, "#bdc3c7") for k in counts.index]
    axes[1].pie(
        counts.values,
        labels=counts.index,
        colors=wedge_colors,
        autopct="%1.1f%%",
        startangle=90,
        pctdistance=0.8,
        wedgeprops={"linewidth": 2, "edgecolor": "white"},
    )
    axes[1].set_title("Overall Pipeline Recovery Rate", fontweight="bold")

    plt.suptitle(
        f"LangGraph RAG vs Original AI Errors  (n={len(df_cmp)} feature-level errors)",
        fontsize=13, fontweight="bold",
    )
    plt.tight_layout()
    save_path = REPORTS_DIR / "pipeline_recovery_vs_original_errors.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

## 10. Figure — Recovery Rate by Feature

In [ ]:
if "recovery_status" not in df_cmp.columns:
    print("Run cell 8 first.")
else:
    feat_recovery = (
        df_cmp[df_cmp["recovery_status"] != "NOT_RUN"]
        .groupby("feature_name")["recovery_status"]
        .value_counts()
        .unstack(fill_value=0)
    )

    # Add recovery rate
    feat_recovery["total"] = feat_recovery.sum(axis=1)
    if "RECOVERED" in feat_recovery.columns:
        feat_recovery["recovery_rate"] = feat_recovery["RECOVERED"] / feat_recovery["total"]
    else:
        feat_recovery["recovery_rate"] = 0.0

    feat_recovery = feat_recovery.sort_values("recovery_rate")
    feat_recovery.index = (
        feat_recovery.index
        .str.replace("feature_", "")
        .str.replace("_", " ")
        .str.title()
    )

    status_cols = [s for s in ["RECOVERED", "UNCERTAIN", "CONFIRMED_ERROR", "DIFFERENT_ERROR"]
                   if s in feat_recovery.columns]

    fig, ax = plt.subplots(figsize=(13, 6))
    feat_recovery[status_cols].plot(
        kind="barh", stacked=True,
        color=[rc_colors.get(s, "#bdc3c7") for s in status_cols],
        ax=ax, edgecolor="white", linewidth=1.2,
    )
    ax.axvline(x=feat_recovery["total"].max() / 2,
               color="black", linestyle="--", alpha=0.3)
    ax.set_title(
        "Pipeline Recovery by Clinical Feature\n"
        "(Only features with original AI fabrication or omission)",
        fontweight="bold", fontsize=12,
    )
    ax.set_xlabel("Feature-Level Error Count")
    ax.legend(title="Pipeline Result", loc="lower right", fontsize=9)
    plt.tight_layout()
    save_path = REPORTS_DIR / "recovery_by_feature.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")
    print("\nRecovery rate by feature:")
    print(feat_recovery[["total", "RECOVERED", "recovery_rate"]]
          .sort_values("recovery_rate", ascending=False)
          .to_string())

## 11. Figure — Verification Confidence for Fabrications vs Omissions

In [ ]:
if df_cmp.empty:
    print("No data.")
else:
    vc_data = df_cmp.dropna(subset=["verification_confidence", "original_ai_error"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Violin: verification confidence by original error type
    if len(vc_data) > 0:
        sns.violinplot(
            data=vc_data, x="original_ai_error", y="verification_confidence",
            palette={"FABRICATION": "#e74c3c", "OMISSION": "#f39c12"},
            inner="box", cut=0, ax=axes[0],
        )
        axes[0].axhline(0.8, color="black", linestyle="--",
                        alpha=0.5, label="Pass threshold")
        axes[0].set_title("Verification Confidence\nby Original Error Type",
                          fontweight="bold")
        axes[0].set_ylabel("Verification Confidence")
        axes[0].set_xlabel("Original AI Error")
        axes[0].legend()

    # Scatter: extraction confidence vs verification confidence
    conf_data = df_cmp.dropna(subset=["pipeline_confidence", "verification_confidence"])
    if len(conf_data) > 0:
        for orig_err, grp in conf_data.groupby("original_ai_error"):
            color = "#e74c3c" if orig_err == "FABRICATION" else "#f39c12"
            axes[1].scatter(
                grp["pipeline_confidence"], grp["verification_confidence"],
                label=orig_err, color=color, alpha=0.7, s=60,
                edgecolors="white", linewidths=0.5,
            )
        axes[1].axhline(0.8, color="red",  linestyle="--", alpha=0.4)
        axes[1].axvline(0.75, color="blue", linestyle="--", alpha=0.4)
        axes[1].set_xlabel("Pipeline Extraction Confidence")
        axes[1].set_ylabel("Verification Confidence")
        axes[1].set_title("Extraction vs Verification Confidence\n(error cases only)",
                          fontweight="bold")
        axes[1].legend(title="Original Error")

    plt.suptitle("Confidence Analysis — Fabrication & Omission Cases",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    save_path = REPORTS_DIR / "verification_confidence_error_cases.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

## 12. Auditable Case-Level Detail Table

In [ ]:
if df_cmp.empty:
    print("No data.")
else:
    display_cols = [
        "mrn", "patient_initials", "surgeon",
        "feature_name", "original_ai_error",
        "pipeline_verdict", "recovery_status",
        "pipeline_value", "pipeline_confidence",
        "pipeline_supported", "verification_confidence",
        "verification_quote", "evidence",
    ]
    display_cols = [c for c in display_cols if c in df_cmp.columns]

    detail = df_cmp[display_cols].copy()
    detail["feature_name"] = (
        detail["feature_name"]
        .str.replace("feature_", "")
        .str.replace("_", " ")
        .str.title()
    )

    # Sort: recovered first, then confirmed errors, then uncertain
    order = {"RECOVERED": 0, "CONFIRMED_ERROR": 1, "UNCERTAIN": 2,
             "DIFFERENT_ERROR": 3, "NOT_RUN": 4}
    if "recovery_status" in detail.columns:
        detail["_sort"] = detail["recovery_status"].map(order).fillna(5)
        detail = detail.sort_values(["_sort", "mrn", "feature_name"]).drop(columns=["_sort"])

    # Save full audit CSV
    audit_path = RUN_OUT_DIR / "audit_table.csv"
    detail.to_csv(audit_path, index=False)
    print(f"Full audit table saved: {audit_path}")
    print(f"Rows: {len(detail)}")
    print()

    # Preview fabrications that were CONFIRMED by pipeline
    confirmed_fabs = detail[
        (detail["original_ai_error"] == "FABRICATION") &
        (detail.get("recovery_status", pd.Series()) == "CONFIRMED_ERROR")
    ] if "recovery_status" in detail.columns else detail[detail["original_ai_error"] == "FABRICATION"]

    print(f"=== CONFIRMED FABRICATIONS (n={len(confirmed_fabs)}) ===")
    if len(confirmed_fabs) > 0:
        print(confirmed_fabs[[
            "mrn", "feature_name", "pipeline_value",
            "pipeline_confidence", "verification_quote"
        ]].to_string(index=False))

## 13. Feature-Level Error Heatmap — All 63 Error Cases

In [ ]:
# Build a patient × feature heatmap of error types from validation sheet
heatmap_rows = []
for _, row in df_bad.iterrows():
    mrn = int(row["mrn"])
    label = f"{row['patient_initials']} ({row['surgeon'].split(',')[0]})"
    feat_vals = {}
    for col, feat in COL_TO_FEATURE.items():
        if col in row.index:
            val = row[col]
            feat_vals[feat] = int(val) if pd.notna(val) else 0
    feat_vals["label"] = label
    feat_vals["mrn"]   = mrn
    heatmap_rows.append(feat_vals)

hm_df = pd.DataFrame(heatmap_rows).drop_duplicates("mrn")
hm_df = hm_df.set_index("label")
feat_cols = [c for c in hm_df.columns if c.startswith("feature_")]
hm_data = hm_df[feat_cols].fillna(0).astype(int)
hm_data.columns = [
    c.replace("feature_","").replace("_"," ").title()[:22]
    for c in hm_data.columns
]

fig_h = max(8, len(hm_data) * 0.35)
fig, ax = plt.subplots(figsize=(14, fig_h))

from matplotlib.colors import ListedColormap
cmap = ListedColormap(["#f8f9fa", "#2ecc71", "#f39c12", "#e74c3c"])

im = ax.imshow(hm_data.values, cmap=cmap, vmin=0, vmax=3, aspect="auto")
ax.set_xticks(range(len(hm_data.columns)))
ax.set_xticklabels(hm_data.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(hm_data.index)))
ax.set_yticklabels(hm_data.index, fontsize=8)

# Annotate cells with error type
val_map = {0: "", 1: "✓", 2: "O", 3: "F"}
for i in range(len(hm_data.index)):
    for j in range(len(hm_data.columns)):
        v = hm_data.values[i, j]
        if v in (2, 3):
            ax.text(j, i, val_map[v], ha="center", va="center",
                    fontsize=8, fontweight="bold",
                    color="white" if v == 3 else "#333333")

cbar = plt.colorbar(im, ax=ax, shrink=0.4, ticks=[0,1,2,3])
cbar.ax.set_yticklabels(["Not scored", "Correct", "Omission", "Fabrication"])
ax.set_title(
    "AI Error Heatmap — Cases with Fabrications or Omissions\n"
    "O=Omission  F=Fabrication",
    fontweight="bold", fontsize=12,
)
plt.tight_layout()
save_path = REPORTS_DIR / "error_heatmap_all_cases.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

## 14. Save Final Summary Report

In [ ]:
import json
from datetime import datetime

summary = {
    "run_timestamp": datetime.utcnow().isoformat(),
    "run_id": run_id,
    "model_id": MODEL_ID,
    "prompt_id": PROMPT_ID,
    "dry_run": DRY_RUN,
    "input": {
        "total_validation_cases": len(df_val),
        "cases_with_ai_errors": int(bad_mask.sum()),
        "fabrication_cases": int(fab_mask.sum()),
        "omission_cases": int(omit_mask.sum()),
        "total_pdfs": sum(pdf_counts.values()),
        "patients_with_ocr": len(ok_patients),
    },
    "pipeline": {
        "patients_processed": len(all_results),
        "patients_failed": len(failed_cases),
        "feature_errors_rerun": len(df_cmp),
    },
}

if "recovery_status" in df_cmp.columns and not df_cmp.empty:
    rc = df_cmp["recovery_status"].value_counts()
    total_rc = len(df_cmp[df_cmp["recovery_status"] != "NOT_RUN"])
    summary["recovery"] = {
        "total_evaluated": total_rc,
        "recovered": int(rc.get("RECOVERED", 0)),
        "confirmed_error": int(rc.get("CONFIRMED_ERROR", 0)),
        "uncertain": int(rc.get("UNCERTAIN", 0)),
        "recovery_rate": round(rc.get("RECOVERED", 0) / total_rc, 3) if total_rc else None,
    }

summary_path = RUN_OUT_DIR / "run_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 60)
print("RUN SUMMARY")
print("=" * 60)
print(json.dumps(summary, indent=2))
print(f"\nAll outputs in: {RUN_OUT_DIR}")
print(f"Figures in    : {REPORTS_DIR}")

---
## 15. Deblur Pipeline — Improve OCR on Blurry Source PDFs

Re-renders every PDF from the 63 error cases, scores each page for blur using
Laplacian variance, and applies a full deblur + contrast-enhancement pipeline
to pages that fall below the blur threshold. Re-OCRs the deblurred images and
compares word counts to the original cached OCR output.

**Classification per PDF:**
- `SHARP` — all pages above blur threshold; no deblur needed
- `BLURRY_RECOVERED` — blurry + original OCR failed (< 50 words) → deblur succeeded (≥ 50 words)
- `BLURRY_STILL_FAILED` — blurry + original OCR failed → deblur still produced < 50 words
- `BLURRY_IMPROVED` — blurry + original OCR had some text → deblur meaningfully increased word count
- `BLURRY_NO_GAIN` — blurry + deblur produced no meaningful improvement

**Outputs:**
- `reports/deblur_ocr_comparison.csv` — per-PDF before/after stats
- `reports/deblur_improvement.png` — visualisations
- `data_private/ocr_cache_deblurred/{stem}.txt` — deblurred OCR text cached for re-use

In [ ]:
import cv2
import numpy as np
from PIL import Image

# ── Deblur config ─────────────────────────────────────────────────────────────
BLUR_THRESHOLD   = 100    # Laplacian variance; below this → blurry page
DEBLUR_DPI       = 200    # match the OCR DPI already used above
LOW_WORD_THRESH  = 50     # original OCR word count below this = "failed extraction"
WORD_GAIN_THRESH = 30     # minimum extra words to count as "recovered" or "improved"

OCR_DEBLUR_CACHE = DATA_PRIVATE / "ocr_cache_deblurred"
OCR_DEBLUR_CACHE.mkdir(parents=True, exist_ok=True)

# ── Core functions (adapted from notebooks/08_ocr_image_quality_deblur.ipynb) ─

def detect_blur(image: np.ndarray, threshold: float = BLUR_THRESHOLD) -> tuple:
    """Laplacian variance blur detector. Returns (is_blurry: bool, score: float)."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    return score < threshold, score


def deblur_image(image: np.ndarray) -> np.ndarray:
    """Contrast, upscale, denoise, dual-threshold, sharpen, morph cleanup."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image.copy()
    gray = cv2.equalizeHist(gray)
    gray = cv2.convertScaleAbs(gray, alpha=2.5, beta=50)
    h, w = gray.shape
    gray = cv2.resize(gray, (w * 2, h * 2), interpolation=cv2.INTER_CUBIC)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    _, bin_otsu   = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bin_adapt     = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
    )
    binary = cv2.bitwise_and(bin_adapt, bin_otsu)
    kernel_sharp = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    binary = cv2.filter2D(binary, -1, kernel_sharp)
    kernel = np.ones((3, 3), np.uint8)
    binary = cv2.dilate(binary, kernel, iterations=1)
    binary = cv2.erode(binary,  kernel, iterations=1)
    return binary


def pdf_to_images_np(pdf_path: Path, dpi: int = DEBLUR_DPI) -> list:
    """Convert PDF pages to numpy BGR arrays via PyMuPDF."""
    doc    = fitz.open(str(pdf_path))
    mat    = fitz.Matrix(dpi / 72, dpi / 72)
    images = []
    for page in doc:
        pix = page.get_pixmap(matrix=mat, alpha=False)
        arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
            pix.height, pix.width, pix.n
        )
        arr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR) if pix.n == 3 else \
              cv2.cvtColor(arr, cv2.COLOR_RGBA2BGR)
        images.append(arr)
    doc.close()
    return images


def ocr_image_np(image: np.ndarray) -> str:
    """Tesseract OCR on a numpy image. Returns cleaned text."""
    pil_img = Image.fromarray(image)
    raw = pytesseract.image_to_string(pil_img, config="--oem 3 --psm 6 --dpi 300")
    return re.sub(r'\s+', ' ', raw).strip()


def word_count(text: str) -> int:
    return len(text.split()) if text else 0


print("Deblur functions defined.")
print(f"Blur threshold  : {BLUR_THRESHOLD}")
print(f"Deblur DPI      : {DEBLUR_DPI}")
print(f"Low-word cutoff : {LOW_WORD_THRESH} words  (below = 'failed extraction')")
print(f"Word-gain min   : {WORD_GAIN_THRESH} words  (above = meaningful improvement)")
print(f"Deblur cache    : {OCR_DEBLUR_CACHE}")

In [ ]:
# ── Step 1: Blur scan across all PDFs from the 63 error cases ─────────────────
# Renders first 3 pages of each PDF and computes Laplacian variance blur score.
# Also reads word count from the existing OCR cache to identify failed extractions.

MAX_PAGES_BLUR = 3  # pages to check per PDF for blur detection

blur_scan_rows = []
print(f"Scanning {len(all_pdfs)} PDFs for blur...\n")

for pdf_path in tqdm(all_pdfs, desc="Blur scan"):
    # Original cached OCR word count
    cache_file   = OCR_CACHE_DIR / (pdf_path.stem + ".txt")
    cached_text  = cache_file.read_text(encoding="utf-8") if cache_file.exists() else ""
    words_cached = word_count(cached_text)

    try:
        images = pdf_to_images_np(pdf_path, dpi=DEBLUR_DPI)
        pages_to_check = images[:MAX_PAGES_BLUR] if MAX_PAGES_BLUR else images

        page_scores = []
        for img in pages_to_check:
            _, score = detect_blur(img)
            page_scores.append(score)

        min_score  = min(page_scores)
        mean_score = float(np.mean(page_scores))
        pdf_blurry = min_score < BLUR_THRESHOLD   # blurry if ANY page is below threshold

        blur_scan_rows.append({
            "pdf_path":         str(pdf_path),
            "pdf_stem":         pdf_path.stem,
            "surgeon_folder":   pdf_path.parent.parent.name,
            "case_folder":      pdf_path.parent.name,
            "n_pages_checked":  len(page_scores),
            "blur_score_min":   round(min_score, 2),
            "blur_score_mean":  round(mean_score, 2),
            "is_blurry":        pdf_blurry,
            "words_original":   words_cached,
            "ocr_cache_exists": cache_file.exists(),
            "extraction_failed_original": words_cached < LOW_WORD_THRESH,
        })
    except Exception as exc:
        blur_scan_rows.append({
            "pdf_path":         str(pdf_path),
            "pdf_stem":         pdf_path.stem,
            "surgeon_folder":   pdf_path.parent.parent.name,
            "case_folder":      pdf_path.parent.name,
            "n_pages_checked":  0,
            "blur_score_min":   None,
            "blur_score_mean":  None,
            "is_blurry":        None,
            "words_original":   words_cached,
            "ocr_cache_exists": cache_file.exists(),
            "extraction_failed_original": words_cached < LOW_WORD_THRESH,
            "scan_error":       str(exc),
        })

df_blur_scan = pd.DataFrame(blur_scan_rows)

# Summary
n_blurry  = df_blur_scan["is_blurry"].sum()
n_sharp   = (~df_blur_scan["is_blurry"].fillna(False)).sum()
n_failed  = df_blur_scan["extraction_failed_original"].sum()
n_blurry_and_failed = (
    df_blur_scan["is_blurry"].fillna(False) &
    df_blur_scan["extraction_failed_original"]
).sum()

print(f"Total PDFs scanned          : {len(df_blur_scan)}")
print(f"  Sharp  (score >= {BLUR_THRESHOLD})    : {n_sharp}")
print(f"  Blurry (score <  {BLUR_THRESHOLD})    : {n_blurry}")
print(f"  Original OCR failed (<{LOW_WORD_THRESH}w) : {n_failed}")
print(f"  Blurry AND extraction failed: {n_blurry_and_failed}  <- deblur candidates")
print()
print("Blur score distribution (blurry PDFs):")
print(df_blur_scan[df_blur_scan["is_blurry"] == True]["blur_score_min"].describe().round(2))

In [ ]:
# ── Step 2: Deblur + Re-OCR all blurry PDFs ──────────────────────────────────
# For each blurry PDF: deblur every page, re-OCR, cache result.
# Also processes sharp PDFs with failed extraction (text-layer PDFs may have
# rendering issues unrelated to image blur).

candidates = df_blur_scan[
    df_blur_scan["is_blurry"].fillna(False) |          # blurry
    df_blur_scan["extraction_failed_original"]          # or original OCR failed
]["pdf_stem"].tolist()

print(f"Deblur candidates : {len(candidates)} PDFs")
print(f"  (already in deblur cache: "
      f"{sum(1 for s in candidates if (OCR_DEBLUR_CACHE / (s + '.txt')).exists())})")
print()

deblur_results = {}   # stem → {"words_deblurred": int, "text": str}

for pdf_stem in tqdm(candidates, desc="Deblur+OCR"):
    cache_out = OCR_DEBLUR_CACHE / (pdf_stem + ".txt")

    # Use cached deblur result if available
    if cache_out.exists():
        text = cache_out.read_text(encoding="utf-8")
        deblur_results[pdf_stem] = {"words_deblurred": word_count(text), "text": text}
        continue

    # Find the PDF path
    match = df_blur_scan[df_blur_scan["pdf_stem"] == pdf_stem]
    if match.empty:
        continue
    pdf_path = Path(match.iloc[0]["pdf_path"])
    if not pdf_path.exists():
        deblur_results[pdf_stem] = {"words_deblurred": 0, "text": "", "error": "file_missing"}
        continue

    try:
        images = pdf_to_images_np(pdf_path, dpi=DEBLUR_DPI)
        page_texts = []
        for page_num, img in enumerate(images, start=1):
            img_db   = deblur_image(img)
            page_txt = ocr_image_np(img_db)
            page_texts.append(f"[PAGE {page_num}]\n{page_txt}")

        full_text = "\n\n".join(page_texts)
        cache_out.write_text(full_text, encoding="utf-8")
        deblur_results[pdf_stem] = {
            "words_deblurred": word_count(full_text),
            "text": full_text,
        }
    except Exception as exc:
        deblur_results[pdf_stem] = {"words_deblurred": 0, "text": "", "error": str(exc)}

print(f"\nDeblur complete for {len(deblur_results)} PDFs")
print(f"Errors           : {sum(1 for v in deblur_results.values() if 'error' in v)}")

In [ ]:
# ── Step 3: Classify each PDF and build comparison table ─────────────────────

def classify_deblur_outcome(row: pd.Series, deblur_res: dict) -> str:
    """
    SHARP               — not blurry; original OCR adequate
    SHARP_FAILED        — not blurry but extraction still failed (non-image issue)
    BLURRY_RECOVERED    — blurry + original failed + deblur succeeded
    BLURRY_STILL_FAILED — blurry + original failed + deblur still failed
    BLURRY_IMPROVED     — blurry + had some text + deblur gave meaningful gain
    BLURRY_NO_GAIN      — blurry + deblur gave no meaningful improvement
    DEBLUR_ERROR        — deblur processing failed
    """
    is_blurry = bool(row.get("is_blurry", False))
    orig_fail = bool(row.get("extraction_failed_original", False))
    stem      = row["pdf_stem"]
    dr        = deblur_res.get(stem, {})
    words_db  = dr.get("words_deblurred", 0)
    words_orig = int(row.get("words_original", 0))

    if "error" in dr and not is_blurry:
        return "SHARP"

    if not is_blurry:
        if orig_fail:
            return "SHARP_FAILED"
        return "SHARP"

    if "error" in dr:
        return "DEBLUR_ERROR"

    gain = words_db - words_orig

    if orig_fail:
        if words_db >= LOW_WORD_THRESH:
            return "BLURRY_RECOVERED"
        else:
            return "BLURRY_STILL_FAILED"
    else:
        if gain >= WORD_GAIN_THRESH:
            return "BLURRY_IMPROVED"
        else:
            return "BLURRY_NO_GAIN"


comparison_rows = []
for _, row in df_blur_scan.iterrows():
    stem    = row["pdf_stem"]
    dr      = deblur_results.get(stem, {})
    outcome = classify_deblur_outcome(row, deblur_results)

    comparison_rows.append({
        "pdf_stem":             stem,
        "pdf_path":             row["pdf_path"],
        "surgeon_folder":       row.get("surgeon_folder"),
        "case_folder":          row.get("case_folder"),
        "blur_score_min":       row.get("blur_score_min"),
        "blur_score_mean":      row.get("blur_score_mean"),
        "is_blurry":            row.get("is_blurry"),
        "words_original":       int(row.get("words_original", 0)),
        "words_deblurred":      dr.get("words_deblurred", None),
        "word_gain":            (dr.get("words_deblurred", 0) - int(row.get("words_original", 0)))
                                if stem in deblur_results else None,
        "extraction_failed_original": row.get("extraction_failed_original"),
        "extraction_failed_deblurred": (
            dr.get("words_deblurred", 0) < LOW_WORD_THRESH
            if stem in deblur_results else None
        ),
        "outcome": outcome,
    })

df_deblur_cmp = pd.DataFrame(comparison_rows)

# ── Print classification summary ─────────────────────────────────────────────
outcome_counts = df_deblur_cmp["outcome"].value_counts()
print("=" * 60)
print("DEBLUR OUTCOME CLASSIFICATION")
print("=" * 60)
for outcome, count in outcome_counts.items():
    pct = count / len(df_deblur_cmp) * 100
    print(f"  {outcome:<28} {count:>4}  ({pct:.1f}%)")

print()
n_recovered = outcome_counts.get("BLURRY_RECOVERED", 0)
n_failed    = outcome_counts.get("BLURRY_STILL_FAILED", 0)
n_improved  = outcome_counts.get("BLURRY_IMPROVED", 0)
n_blurry    = df_deblur_cmp["is_blurry"].fillna(False).sum()

print(f"Of {n_blurry} blurry PDFs:")
print(f"  Recovered (was failed → now usable)  : {n_recovered}")
print(f"  Still failed after deblur            : {n_failed}")
print(f"  Improved (had text, further gained)  : {n_improved}")

# ── Detailed lists ────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("BLURRY + RECOVERED (deblur restored extractable text):")
print("=" * 60)
recovered_df = df_deblur_cmp[df_deblur_cmp["outcome"] == "BLURRY_RECOVERED"][
    ["pdf_stem", "case_folder", "blur_score_min", "words_original", "words_deblurred", "word_gain"]
]
if len(recovered_df):
    print(recovered_df.to_string(index=False))
else:
    print("  None")

print()
print("=" * 60)
print("BLURRY + STILL FAILED (deblur could not recover text):")
print("=" * 60)
still_failed_df = df_deblur_cmp[df_deblur_cmp["outcome"] == "BLURRY_STILL_FAILED"][
    ["pdf_stem", "case_folder", "blur_score_min", "words_original", "words_deblurred"]
]
if len(still_failed_df):
    print(still_failed_df.to_string(index=False))
else:
    print("  None")

# ── Save CSV ──────────────────────────────────────────────────────────────────
save_path = REPORTS_DIR / "deblur_ocr_comparison.csv"
df_deblur_cmp.to_csv(save_path, index=False)
print(f"\nFull comparison table saved: {save_path}  ({len(df_deblur_cmp)} rows)")

In [ ]:
# ── Step 4: Figures ───────────────────────────────────────────────────────────

OUTCOME_COLORS = {
    "SHARP":             "#2ecc71",
    "BLURRY_RECOVERED":  "#3498db",
    "BLURRY_IMPROVED":   "#9b59b6",
    "BLURRY_NO_GAIN":    "#f39c12",
    "BLURRY_STILL_FAILED": "#e74c3c",
    "SHARP_FAILED":      "#e67e22",
    "DEBLUR_ERROR":      "#95a5a6",
}

outcome_order = [
    "SHARP", "BLURRY_RECOVERED", "BLURRY_IMPROVED",
    "BLURRY_NO_GAIN", "BLURRY_STILL_FAILED", "SHARP_FAILED", "DEBLUR_ERROR",
]
present_outcomes = [o for o in outcome_order if o in outcome_counts.index]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Panel 1: outcome breakdown (horizontal bar) ───────────────────────────────
ax = axes[0]
counts = [outcome_counts.get(o, 0) for o in present_outcomes]
colors = [OUTCOME_COLORS[o] for o in present_outcomes]
bars = ax.barh(present_outcomes, counts, color=colors, edgecolor="white", linewidth=1.2)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            str(cnt), va="center", fontsize=9)
ax.set_xlabel("Number of PDFs")
ax.set_title("Deblur Outcome Classification\n(all error-case PDFs)", fontweight="bold")
ax.invert_yaxis()

# ── Panel 2: word count before vs after (blurry PDFs only) ───────────────────
ax = axes[1]
blurry_df = df_deblur_cmp[
    df_deblur_cmp["is_blurry"].fillna(False) &
    df_deblur_cmp["words_deblurred"].notna()
].copy()

if not blurry_df.empty:
    ax.scatter(
        blurry_df["words_original"],
        blurry_df["words_deblurred"],
        c=[OUTCOME_COLORS.get(o, "#bdc3c7") for o in blurry_df["outcome"]],
        s=60, edgecolors="white", linewidths=0.5, alpha=0.85,
    )
    max_val = max(blurry_df[["words_original", "words_deblurred"]].max())
    ax.plot([0, max_val], [0, max_val], "k--", alpha=0.3, linewidth=1, label="no change")
    ax.axhline(LOW_WORD_THRESH, color="red",  linestyle=":", linewidth=1.2,
               label=f"recovery threshold ({LOW_WORD_THRESH}w)")
    ax.axvline(LOW_WORD_THRESH, color="gray", linestyle=":", linewidth=1.2)
    ax.set_xlabel("Word Count — Original OCR")
    ax.set_ylabel("Word Count — After Deblur")
    ax.set_title("OCR Word Count:\nOriginal vs After Deblur (blurry PDFs)", fontweight="bold")
    ax.legend(fontsize=8)

    # Colour legend
    for outcome, color in OUTCOME_COLORS.items():
        if outcome in blurry_df["outcome"].values:
            ax.scatter([], [], c=color, s=40,
                       label=outcome.replace("_", " ").title())
    ax.legend(fontsize=7, loc="upper left", title="Outcome")
else:
    ax.text(0.5, 0.5, "No blurry PDFs found", transform=ax.transAxes, ha="center")

# ── Panel 3: word gain distribution for blurry PDFs ──────────────────────────
ax = axes[2]
gain_data = df_deblur_cmp[
    df_deblur_cmp["is_blurry"].fillna(False) &
    df_deblur_cmp["word_gain"].notna()
]["word_gain"]

if not gain_data.empty:
    ax.hist(gain_data[gain_data >= 0],  bins=25, color="#3498db", alpha=0.75,
            edgecolor="white", label="Gain (positive)")
    ax.hist(gain_data[gain_data < 0],   bins=10, color="#e74c3c", alpha=0.75,
            edgecolor="white", label="Loss (negative)")
    ax.axvline(0,                  color="black", linestyle="--", linewidth=1.2)
    ax.axvline(WORD_GAIN_THRESH,   color="green", linestyle=":",  linewidth=1.2,
               label=f"Gain threshold ({WORD_GAIN_THRESH}w)")
    ax.set_xlabel("Word Count Change (after − before)")
    ax.set_ylabel("Number of PDFs")
    ax.set_title("Distribution of Word Count Gain\nAfter Deblurring (blurry PDFs)",
                 fontweight="bold")
    ax.legend(fontsize=8)
    improved_pct = (gain_data >= WORD_GAIN_THRESH).mean() * 100
    ax.text(0.97, 0.97, f"{improved_pct:.0f}% improved\n(gain ≥ {WORD_GAIN_THRESH}w)",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.7))
else:
    ax.text(0.5, 0.5, "No blurry PDFs", transform=ax.transAxes, ha="center")

plt.suptitle(
    f"Deblur Pipeline Results — {len(all_pdfs)} PDFs from {len(patient_pdfs)} Error Cases",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
save_path = REPORTS_DIR / "deblur_improvement.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

In [ ]:
# ── Step 5: Rebuild patient OCR using deblurred text where it improved ────────
# For each patient, replace per-PDF OCR text with deblurred version wherever
# the deblur produced meaningfully more words (BLURRY_RECOVERED or BLURRY_IMPROVED).
# Patients with at least one recovered PDF are flagged for pipeline re-run.

IMPROVED_OUTCOMES = {"BLURRY_RECOVERED", "BLURRY_IMPROVED"}

# Build a lookup: pdf_stem → deblurred text (only for improved outcomes)
deblur_upgrade = {}
for _, row in df_deblur_cmp.iterrows():
    if row["outcome"] in IMPROVED_OUTCOMES:
        stem = row["pdf_stem"]
        dr   = deblur_results.get(stem, {})
        if dr.get("text"):
            deblur_upgrade[stem] = dr["text"]

print(f"PDFs with deblur upgrades : {len(deblur_upgrade)}")
print()

# Rebuild patient_ocr_deblurred — same structure as patient_ocr (cell 10)
patient_ocr_deblurred: dict = {}
patients_with_upgrade: list = []

for mrn, pdf_paths in patient_pdfs.items():
    sorted_pdfs = sorted(pdf_paths, key=_doc_sort_key)
    sections = []
    upgraded_count = 0
    for pdf in sorted_pdfs:
        stem = pdf.stem
        if stem in deblur_upgrade:
            text = deblur_upgrade[stem]
            upgraded_count += 1
            sections.append(f"=== SOURCE (DEBLURRED): {pdf.name} ===\n{text}")
        else:
            text = ocr_results.get(pdf, "")
            if text and not text.startswith("[OCR_ERROR") and not text.startswith("[FUTURE_ERROR"):
                sections.append(f"=== SOURCE: {pdf.name} ===\n{text}")

    patient_ocr_deblurred[mrn] = "\n\n".join(sections)
    if upgraded_count > 0:
        patients_with_upgrade.append(mrn)

ok_patients_deblurred = {
    mrn for mrn, txt in patient_ocr_deblurred.items() if len(txt) > 200
}

print(f"Patients with at least 1 deblurred PDF : {len(patients_with_upgrade)}")
print(f"  -> These patients should be re-run through the pipeline")
print(f"     (set patient_ocr = patient_ocr_deblurred and clear their cached results)")
print()

# Print per-patient upgrade summary
if patients_with_upgrade:
    upgrade_summary = []
    for mrn in patients_with_upgrade:
        info  = patient_info.get(mrn, {})
        pdfs  = patient_pdfs.get(mrn, [])
        n_upg = sum(1 for p in pdfs if p.stem in deblur_upgrade)
        orig_len = len(patient_ocr.get(mrn, ""))
        new_len  = len(patient_ocr_deblurred.get(mrn, ""))
        upgrade_summary.append({
            "mrn":             mrn,
            "initials":        info.get("patient_initials"),
            "surgeon":         info.get("surgeon_last"),
            "n_pdfs_upgraded": n_upg,
            "chars_original":  orig_len,
            "chars_deblurred": new_len,
            "char_gain":       new_len - orig_len,
        })
    upg_df = pd.DataFrame(upgrade_summary).sort_values("char_gain", ascending=False)
    print("Patients with deblur upgrades (sorted by character gain):")
    print(upg_df.to_string(index=False))
    upg_df.to_csv(REPORTS_DIR / "deblur_patient_upgrades.csv", index=False)
    print(f"\nSaved: {REPORTS_DIR / 'deblur_patient_upgrades.csv'}")
else:
    print("No patients received a deblur upgrade.")

# ── Final deblur summary ──────────────────────────────────────────────────────
print()
print("=" * 60)
print("DEBLUR PIPELINE — FINAL SUMMARY")
print("=" * 60)
print(f"Total PDFs scanned                 : {len(df_deblur_cmp)}")
print(f"Sharp PDFs (no action needed)      : {outcome_counts.get('SHARP', 0)}")
print(f"Blurry PDFs total                  : {int(df_deblur_cmp['is_blurry'].fillna(False).sum())}")
print(f"  -> RECOVERED (failed -> usable)  : {outcome_counts.get('BLURRY_RECOVERED', 0)}")
print(f"  -> IMPROVED  (text gain >= {WORD_GAIN_THRESH}w) : {outcome_counts.get('BLURRY_IMPROVED', 0)}")
print(f"  -> STILL FAILED after deblur     : {outcome_counts.get('BLURRY_STILL_FAILED', 0)}")
print(f"  -> NO GAIN from deblur           : {outcome_counts.get('BLURRY_NO_GAIN', 0)}")
print(f"Sharp but extraction failed        : {outcome_counts.get('SHARP_FAILED', 0)}")
print(f"Deblur processing errors           : {outcome_counts.get('DEBLUR_ERROR', 0)}")
print()
print(f"Patients flagged for pipeline re-run : {len(patients_with_upgrade)}")
print()
print("To re-run the pipeline with deblurred OCR:")
print("  1. Use patient_ocr_deblurred instead of patient_ocr in cell 15")
print("  2. Clear cached results for affected MRNs from pipeline_results.json")
print("  3. Re-run cells 15 → 14 (run_id will be updated automatically)")